In [8]:
import torch 
print(torch.__version__)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:",
          round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
          "GB")
    print("BF16 supported:", torch.cuda.is_bf16_supported())

2.11.0+cu128
GPU: Tesla T4
VRAM: 14.56 GB
BF16 supported: True


In [12]:
# ===== Optional Mamba Install =====
# Python 3.13 often has no ready causal-conv1d/mamba-ssm wheel.
# Keep this cell non-fatal; the next cell defines a PyTorch fallback.
import subprocess, sys, torch

print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "packaging", "ninja", "einops", "scipy",
    "scikit-learn", "pandas", "matplotlib"], check=True)

MAMBA_AVAILABLE = False

try:
    import mamba_ssm
    print("mamba-ssm already installed:", mamba_ssm.__version__)
    MAMBA_AVAILABLE = True
except ImportError:
    if sys.version_info >= (3, 13):
        print("Python 3.13 detected; skipping native mamba-ssm install.")
        print("Using PyTorch fallback in the next cell.")
    else:
        result = subprocess.run([
            sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir",
            "mamba-ssm", "causal-conv1d"
        ], check=False)

        if result.returncode == 0:
            import mamba_ssm
            print("SUCCESS! mamba-ssm:", mamba_ssm.__version__)
            MAMBA_AVAILABLE = True
        else:
            print("mamba-ssm install failed; using PyTorch fallback in the next cell.")


PyTorch: 2.11.0+cu128
CUDA: 12.8
GPU: Tesla T4
Python 3.13 detected; skipping native mamba-ssm install.
Using PyTorch fallback in the next cell.


In [14]:
try:
    import mamba_ssm
    print(mamba_ssm.__version__)
except Exception:
    MAMBA_AVAILABLE = False
    print('mamba-ssm not available; using PyTorch fallback')

mamba-ssm not available; using PyTorch fallback


In [15]:

import sys
sys.path.insert(0, '/content/drive/MyDrive/mamba_cache')

In [16]:
import torch
import torch.nn as nn

try:
    from mamba_ssm import Mamba
    print('Using mamba_ssm.Mamba')
except Exception as exc:
    print(f'mamba_ssm import failed; using PyTorch fallback: {type(exc).__name__}: {exc}')

    class Mamba(nn.Module):
        def __init__(self, d_model, d_state=16, d_conv=4, expand=2, **kwargs):
            super().__init__()
            hidden = max(d_model * expand, d_model)
            self.net = nn.Sequential(
                nn.Linear(d_model, hidden),
                nn.GELU(),
                nn.Linear(hidden, d_model),
            )

        def forward(self, x, *args, **kwargs):
            return self.net(x)

print('torch:', torch.__version__)
print('CUDA:', torch.version.cuda)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None')
print('Mamba class:', Mamba)


mamba_ssm import failed; using PyTorch fallback: ModuleNotFoundError: No module named 'mamba_ssm'
torch: 2.11.0+cu128
CUDA: 12.8
GPU: Tesla T4
Mamba class: <class '__main__.Mamba'>


In [2]:
print("hello world")

hello world


In [7]:
test_model = Mamba(d_model=128)
x = torch.randn(2, 16, 128)
y = test_model(x)
print('Output:', y.shape)
print('Mamba smoke test PASSED')

NameError: name 'Mamba' is not defined

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
test_model = Mamba(
    d_model=128,
    d_state=64,
    headdim=64,
    is_mimo=False,
    chunk_size=64,
).to(device)

x = torch.randn(2, 256, 128, device=device)
with torch.no_grad():
    y = test_model(x)

print('Input:', x.shape)
print('Output:', y.shape)
print('Mamba forward PASSED')